In [0]:
from pyspark.sql.functions import *

### Load Silver Layer table

In [0]:
order_df = spark.read.table("retail_project.silver.silver_orders")

customer_df = spark.read.table("retail_project.silver.silver_customer")

order_df.display()
customer_df.display()

## Create Customer Dimension

In [0]:
dim_cust = customer_df.select("customer_id", "customer_name" , "email","city","state","loyalty_points", "status")

dim_cust.display()

In [0]:
dim_cust.write.format("delta").mode("overwrite").saveAsTable("retail_project.gold.gold_dim_customer")

### Create Fact Sales Table 

In [0]:
fact_sales = order_df.join(customer_df , on = "customer_id" , how="left").select(order_df.order_id,
    order_df.order_date,
    order_df.customer_id,
    customer_df.customer_name,
    order_df.product_id,
    order_df.product_name,
    order_df.category,
    order_df.quantity,
    order_df.unit_price,
    order_df.discount,
    order_df.total_amount,
    customer_df.city,
    customer_df.state,
    customer_df.gender,
    customer_df.loyalty_points)

fact_sales =  fact_sales.withColumn("Year" , year("order_date")).withColumn("Month" , month("order_date")).withColumn("Day" , day("order_date"))

fact_sales.display()  

In [0]:
gold_df = fact_sales.groupBy(
    "product_id",
    "product_name",
    "customer_id",
    "customer_name",
    "category",
    "Year",
    "Month",
    "order_date",
    "gender",
    "city",
    "state",
    "loyalty_points"
).agg(
    sum("total_amount").alias("Total_revenue"),
    sum("quantity").alias("Total_quantity"),
    countDistinct("order_id").alias("Total_orders"),
    avg("total_amount").alias("Avg_revenue"),
    avg("quantity").alias("Avg_quantity"),
    avg("loyalty_points").alias("Avg_loyalty_points"),
    min("unit_price").alias("Min_price"),
    max("unit_price").alias("Max_price")
)
gold_df.display()

In [0]:
gold_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.gold.gold_fact_sales")